# Vector Design과 청킹(Chunking) 전략

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import sys

os.environ['HF_HOME'] = '/content/drive/MyDrive/hf_cache'

!pip install -q  sentence-transformers scikit-learn transformers

from sentence_transformers import SentenceTransformer

_MODEL_CACHE = {}

def load_model(name="paraphrase-multilingual-MiniLM-L12-v2"):
    """같은 모델을 여러 섹션에서 재로딩하지 않도록 캐싱한다."""
    if name not in _MODEL_CACHE:
        _MODEL_CACHE[name] = SentenceTransformer(name)
    return _MODEL_CACHE[name]

---

## Vector Design — 벡터와 청크(Chunk)

- 실습 내용
  - 문자를 숫자로 바꿔 보기
  - 임베딩 모델로 문장을 벡터로 변환
  - 문서 → 청크 → 벡터 흐름 실습

In [ ]:
def cos(a, b):
    a, b = np.array(a), np.array(b)
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

In [ ]:
import numpy as np
model = load_model("paraphrase-multilingual-MiniLM-L12-v2")

### [단계 1] 인식의 차이 — 컴퓨터는 숫자만 이해
- 문자 하나하나도 내부적으로는 숫자 코드

In [ ]:
word = "김치"
print([ord(c) for c in word])          # 문자 → 숫자 코드
print(model.encode(word)[:5])          # 문장 → 의미 벡터(앞 5개)

### [단계 2] 분할의 필요성
- 문서 전체를 벡터 하나로 만들면 세부 의미가 뭉개짐
- 모델 입력 길이 제한도 존재

In [ ]:
doc = """김치찌개는 신김치와 돼지고기를 넣고 끓인다. 국물이 얼큰하고 밥과 잘 어울린다.
된장찌개는 된장과 두부, 애호박을 넣고 끓인다. 구수한 맛이 특징이다.
비빔밥은 여러 나물과 고추장을 밥에 넣고 비벼 먹는다. 색이 화려하고 영양이 풍부하다."""

print("최대 입력 토큰:", model.max_seq_length)
whole = model.encode(doc)
print("문서 전체 벡터 차원:", whole.shape)

### [단계 3] 청크 도출 후 임베딩
- 문장 단위로 분할 → 청크마다 벡터 생성

In [ ]:
chunks = [s.strip() for s in doc.replace("\n", " ").split(".") if s.strip()]
for i, c in enumerate(chunks):
    print(i, c)

chunk_vecs = model.encode(chunks)
print("\n청크 수:", len(chunks), "/ 벡터 행렬:", chunk_vecs.shape)

### 4. 전체 벡터 vs 청크 벡터 검색 비교

In [ ]:
q = model.encode("두부가 들어간 구수한 찌개")
print("문서 전체와 유사도:", round(cos(q, whole), 3))
for c, v in zip(chunks, chunk_vecs):
    print(round(cos(q, v), 3), c)

---

## Sparse 벡터와 Dense 벡터 비교

- 실습 내용
  - CountVectorizer로 Sparse 벡터 생성
  - 임베딩 모델로 Dense 벡터 생성
  - 같은 질문으로 두 방식 검색 비교

In [ ]:
def cos(a, b):
    a, b = np.array(a), np.array(b)
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

In [ ]:
import numpy as np
model = load_model("paraphrase-multilingual-MiniLM-L12-v2")
from sklearn.feature_extraction.text import CountVectorizer

In [ ]:
docs = ["김치 찌개 돼지고기", "된장 찌개 두부", "비빔밥 나물 고추장", "매운 국물 요리"]

### 1. Sparse 벡터
- 단어 사전을 만들고, 등장한 단어 위치만 1(또는 횟수)
- 대부분 0

In [ ]:
cv = CountVectorizer(token_pattern=r"[가-힣]+")
sparse = cv.fit_transform(docs).toarray()
print("단어 사전:", cv.get_feature_names_out())
print(sparse)
print("0의 비율:", round((sparse == 0).mean(), 2))

### 2. Dense 벡터
- 모든 값이 0이 아닌 실수
- 단어 맥락·의미를 압축

In [ ]:
dense = model.encode(docs)
print("차원:", dense.shape)
print("첫 벡터 앞 8개:", np.round(dense[0][:8], 3))
print("0의 비율:", round((dense == 0).mean(), 2))

### 3. 검색 비교
- 질문 '얼큰한 찌개' → 단어 일치 vs 의미 유사

In [ ]:
q = "얼큰한 찌개"

print("[Sparse] 단어 일치 개수")
qs = cv.transform([q]).toarray()[0]
for d, v in zip(docs, sparse):
    print(int(v @ qs), d)

print("\n[Dense] 의미 유사도")
qd = model.encode(q)
for d, v in zip(docs, dense):
    print(round(cos(qd, v), 3), d)

---

## Vector 구조 — Sparse와 Dense의 공존

- 실습 내용
  - Item 구조 정의
  - Sparse·Dense 점수 계산
  - 가중치 결합으로 하이브리드 검색

In [ ]:
def cos(a, b):
    a, b = np.array(a), np.array(b)
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

In [ ]:
import numpy as np
model = load_model("paraphrase-multilingual-MiniLM-L12-v2")
from sklearn.feature_extraction.text import TfidfVectorizer

### 1. 기본 설계 구조
- 고유 ID + 원천 데이터 + Sparse 벡터 + Dense 벡터

In [ ]:
docs = ["김치찌개 신김치 돼지고기 얼큰", "된장찌개 된장 두부 구수", "비빔밥 나물 고추장", "매운 국물 요리 추천"]
tfidf = TfidfVectorizer(token_pattern=r"[가-힣]+")
sparse_mat = tfidf.fit_transform(docs)
dense_mat = model.encode(docs)

items = [{"id": f"d{i}", "text": docs[i], "sparse": sparse_mat[i], "dense": dense_mat[i]} for i in range(len(docs))]
print(items[0]["id"], items[0]["text"])
print("sparse 차원:", items[0]["sparse"].shape[1], "/ dense 차원:", len(items[0]["dense"]))

### 2. 각 벡터별 점수 계산
- Sparse: TF-IDF 코사인 → 키워드 정확도
- Dense: 임베딩 코사인 → 문맥 의미

In [ ]:
def sparse_scores(q):
    qv = tfidf.transform([q])
    return (sparse_mat @ qv.T).toarray().ravel()

def dense_scores(q):
    qv = model.encode(q)
    return np.array([cos(qv, v) for v in dense_mat])

q = "얼큰한 찌개"
print("sparse:", np.round(sparse_scores(q), 3))
print("dense :", np.round(dense_scores(q), 3))

### 3. 하이브리드 검색
- 두 점수를 0~1로 정규화 후 가중 합산

In [ ]:
def norm(x):
    return (x - x.min()) / (x.max() - x.min() + 1e-9)

def hybrid(q, alpha=0.5):
    score = alpha * norm(sparse_scores(q)) + (1 - alpha) * norm(dense_scores(q))
    order = np.argsort(-score)
    return [(docs[i], round(float(score[i]), 3)) for i in order]

for a in [1.0, 0.0, 0.5]:
    print(f"alpha={a} (1=키워드만, 0=의미만)")
    for row in hybrid(q, a)[:2]:
        print("  ", row)

---

## 벡터 차원(Dimension)과 문장 길이의 독립성

- 실습 내용
  - 문장 길이별 벡터 shape 비교
  - 모델별 차원 비교

In [ ]:
import numpy as np
model = load_model("paraphrase-multilingual-MiniLM-L12-v2")

### 1. 입력 데이터 예시
- 짧은 문장 vs 긴 문장

In [ ]:
short = "김치찌개는 맛있다."
long = "김치찌개는 신김치와 돼지고기를 함께 넣고 오래 끓일수록 맛있다. 국물은 얼큰하고 밥과 잘 어울리며 두부를 넣으면 더욱 좋다."

print("글자 수:", len(short), "vs", len(long))
print("토큰 수:", len(model.tokenizer.tokenize(short)), "vs", len(model.tokenizer.tokenize(long)))

### 2. 벡터 변환 결과
- 길이가 달라도 차원은 동일

In [ ]:
v_short = model.encode(short)
v_long = model.encode(long)
print("짧은 문장 벡터:", v_short.shape)
print("긴 문장 벡터 :", v_long.shape)

### 3. 차원은 모델이 결정
- 모델 스펙(아키텍처)에 고정

In [ ]:
for name in ["paraphrase-multilingual-MiniLM-L12-v2", "all-MiniLM-L6-v2", "all-mpnet-base-v2"]:
    m = load_model(name)
    print(f"{name:40s} 차원 = {m.get_sentence_embedding_dimension()}")

### 4. 아주 긴 문장은 잘림(truncation)
- 차원은 그대로, 뒷부분 정보만 손실

In [ ]:
very_long = long * 30
print("토큰 수:", len(model.tokenizer.tokenize(very_long)), "/ 모델 한도:", model.max_seq_length)
print("벡터 shape:", model.encode(very_long).shape)

---

## 청킹 전략(Chunking Strategy) — Vector 단위 도출

- 실습 내용
  - 구두점·줄바꿈 기준 청킹
  - 의미 흐름 기준 청킹
  - 질문과 청크 유사도 비교

In [ ]:
def cos(a, b):
    a, b = np.array(a), np.array(b)
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

In [ ]:
import numpy as np
model = load_model("paraphrase-multilingual-MiniLM-L12-v2")

In [ ]:
doc = """김치찌개는 신김치를 볶다가 돼지고기를 넣는다. 물을 붓고 20분 끓인다. 마지막에 두부와 파를 넣는다.
비빔밥은 밥 위에 나물을 올린다. 고추장과 참기름을 넣고 비빈다. 계란 프라이를 올리면 좋다."""

### 1. 휴리스틱 기반 청킹
- 구두점(.)·줄바꿈 등 시각적 기호 기준

In [ ]:
def heuristic_chunk(text):
    return [s.strip() for s in text.replace("\n", ".").split(".") if s.strip()]

h_chunks = heuristic_chunk(doc)
for c in h_chunks: print("-", c)

### 2. 의미 기반 청킹
- 인접 문장의 유사도가 낮아지는 지점에서 분할

In [ ]:
def semantic_chunk(sentences, threshold=0.4):
    vecs = model.encode(sentences)
    chunks, cur = [], [sentences[0]]
    for i in range(1, len(sentences)):
        sim = cos(vecs[i-1], vecs[i])
        if sim < threshold:
            chunks.append(" ".join(cur)); cur = []
        cur.append(sentences[i])
    chunks.append(" ".join(cur))
    return chunks

s_chunks = semantic_chunk(h_chunks)
for c in s_chunks: print("-", c)

### 3. 청킹 품질이 검색에 미치는 영향
- 질문: '김치찌개 만드는 순서'

In [ ]:
q = model.encode("김치찌개 만드는 순서")
for name, chunks in [("휴리스틱", h_chunks), ("의미 기반", s_chunks)]:
    vecs = model.encode(chunks)
    best = max(range(len(chunks)), key=lambda i: cos(q, vecs[i]))
    print(f"[{name}] 최고 유사도 {round(cos(q, vecs[best]), 3)}")
    print("   ", chunks[best])

---

## 청크 크기와 성능·정확도의 트레이드오프 관계

- 실습 내용
  - 여러 청크 크기로 문서 분할
  - 질문별 정답 포함 여부(적중률) 측정
  - 크기별 결과 표로 비교

In [ ]:
def cos(a, b):
    a, b = np.array(a), np.array(b)
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

In [ ]:
import numpy as np
model = load_model("paraphrase-multilingual-MiniLM-L12-v2")

In [ ]:
doc = ("김치찌개는 신김치와 돼지고기를 넣고 끓인다. 국물이 얼큰해서 밥과 잘 어울린다. 두부를 넣으면 더 좋다. "
       "된장찌개는 된장과 두부, 애호박을 넣는다. 구수한 맛이 특징이다. 멸치 육수를 쓰면 깊은 맛이 난다. "
       "비빔밥은 나물과 고추장을 밥에 비빈다. 색이 화려하고 영양이 풍부하다. 계란 프라이를 올리면 좋다. "
       "잡채는 당면을 삶아 채소와 볶는다. 간장과 설탕으로 간을 한다. 명절 음식으로 인기가 많다.")

tests = [("된장찌개 육수", "멸치"), ("비빔밥 토핑", "계란"), ("잡채 간", "간장"), ("김치찌개 재료", "돼지고기")]

### 1. 글자 수 기준 청킹 함수

In [ ]:
def chunk_by_size(text, size):
    return [text[i:i+size] for i in range(0, len(text), size)]

for size in [30, 80, 200]:
    print(size, "글자 →", len(chunk_by_size(doc, size)), "개 청크")

### 2. 크기별 적중률·컨텍스트 길이 측정
- 적중: top-1 청크에 정답 단어 포함
- 컨텍스트 길이: LLM에 전달되는 글자 수(응답 지연 요인)

In [ ]:
def evaluate(size):
    chunks = chunk_by_size(doc, size)
    vecs = model.encode(chunks)
    hits = 0
    for q, answer in tests:
        qv = model.encode(q)
        best = max(range(len(chunks)), key=lambda i: cos(qv, vecs[i]))
        hits += answer in chunks[best]
    return hits / len(tests), len(chunks[0])

print(f"{'청크 크기':>8} {'적중률':>6} {'컨텍스트 길이':>10}")
for size in [20, 40, 80, 150, 300]:
    acc, ctx = evaluate(size)
    print(f"{size:>8} {acc:>6.2f} {ctx:>10}")

### 3. 해석
- 너무 작은 청크: 정답 단어가 다른 청크에 있어 누락
- 너무 큰 청크: 정답은 있지만 불필요한 내용까지 전달 → 응답 지연
- 중간 크기가 균형점(Sweet Spot)

---

## 청킹 전략 수립 시 3대 고려사항

- 실습 내용
  - 콘텐츠 유형별 분기
  - 모델 입력 한도 반영
  - LLM 입력 제한과 청크 수 연결

In [ ]:
model = load_model("paraphrase-multilingual-MiniLM-L12-v2")
tok = model.tokenizer

### ① 색인 대상 콘텐츠의 성격
- 긴 문서(요리책) vs 짧은 콘텐츠(한 줄 후기)

In [ ]:
def content_type(text):
    n = len(tok.tokenize(text))
    return "짧은 콘텐츠" if n < 50 else "긴 문서"

review = "김치찌개 얼큰하고 맛있어요"
book = "김치찌개는 신김치를 볶고 돼지고기를 넣은 뒤 물을 부어 끓인다. " * 10
print(content_type(review), "/", content_type(book))

### ② 어떤 임베딩 모델을 쓰는가
- 모델 최대 입력 토큰이 청크 상한

In [ ]:
model_limits = {"paraphrase-multilingual-MiniLM-L12-v2": model.max_seq_length,
                "bert-base (예시)": 512, "gpt 계열 임베딩 (예시)": 8191}
for k, v in model_limits.items():
    print(f"{k:42s} 최대 {v} 토큰")

### ③ 검색 결과를 어떻게 활용하는가
- 단순 검색 / RAG 답변 / 문서 요약
- LLM 입력 제한 = 청크 수 × 청크 크기

In [ ]:
def plan_chunk(purpose, model_limit, llm_limit=4000, top_k=5):
    base = {"검색": 128, "RAG": 256, "요약": 512}[purpose]
    size = min(base, model_limit)
    if size * top_k > llm_limit:
        size = llm_limit // top_k
    return size

for p in ["검색", "RAG", "요약"]:
    print(f"{p}: 청크 {plan_chunk(p, model.max_seq_length)} 토큰 × top_k 5")

### 4. 종합 판단

In [ ]:
def decide(text, purpose):
    ctype = content_type(text)
    size = plan_chunk(purpose, model.max_seq_length)
    if ctype == "짧은 콘텐츠":
        return f"{ctype}: 분할 없이 1개 청크"
    return f"{ctype}: {size} 토큰 단위 분할 (목적={purpose})"

print(decide(review, "검색"))
print(decide(book, "RAG"))

---

## 데이터 맞춤형 청킹 전략 11가지

- 실습 내용
  - Fixed Size / Content Aware / Overlapping
  - Recursive / Semantic / Hierarchical(Parent-Child)
  - LLM이 필요한 전략(Summarization·Extract Q)은 구조만 확인

In [ ]:
def cos(a, b):
    a, b = np.array(a), np.array(b)
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

In [ ]:
import numpy as np
model = load_model("paraphrase-multilingual-MiniLM-L12-v2")

In [ ]:
doc = """# 김치찌개
신김치를 볶다가 돼지고기를 넣는다. 물을 붓고 20분 끓인다. 두부와 파를 넣는다.

# 비빔밥
밥 위에 나물을 올린다. 고추장과 참기름을 넣고 비빈다. 계란 프라이를 올린다."""

### ① Fixed Size — 고정 글자 수

In [ ]:
def fixed(text, size=30):
    return [text[i:i+size] for i in range(0, len(text), size)]
print(fixed(doc))

### ② Content Aware — 문단(구조) 기준

In [ ]:
def content_aware(text):
    return [p.strip() for p in text.split("\n\n") if p.strip()]
print(content_aware(doc))

### ③ Overlapping — 인접 청크 겹침

In [ ]:
def overlapping(text, size=30, overlap=10):
    return [text[i:i+size] for i in range(0, len(text), size - overlap)]
print(overlapping(doc))

### ④ Recursive — 큰 구분자부터 순서대로 재분할

In [ ]:
def recursive(text, size=25, seps=["\n\n", "\n", ". ", " "]):
    if len(text) <= size or not seps:
        return [text]
    parts = text.split(seps[0])
    out = []
    for p in parts:
        out += recursive(p, size, seps[1:]) if len(p) > size else [p]
    return [c for c in out if c.strip()]
print(recursive(doc))

### ⑥ Semantic — 의미 유사도 기준 병합

In [ ]:
def semantic(text, threshold=0.4):
    sents = [s.strip() for s in text.replace("#", "").replace("\n", ". ").split(". ") if s.strip()]
    vecs = model.encode(sents)
    chunks, cur = [], [sents[0]]
    for i in range(1, len(sents)):
        if cos(vecs[i-1], vecs[i]) < threshold:
            chunks.append(" ".join(cur)); cur = []
        cur.append(sents[i])
    chunks.append(" ".join(cur))
    return chunks
print(semantic(doc))

### ⑦⑪ Hierarchical / Parent-Child — 부모·자식 계층

In [ ]:
def parent_child(text):
    result = []
    for pid, para in enumerate(content_aware(text)):
        for cid, sent in enumerate(s for s in para.split(". ") if s):
            result.append({"parent_id": pid, "child_id": f"{pid}-{cid}", "child": sent, "parent": para})
    return result
for r in parent_child(doc)[:3]:
    print(r["child_id"], "|", r["child"], "→ parent:", r["parent"][:15], "...")

### ⑧⑨⑩ Domain-Specific / Summarization / Extract Candidate Q
- Domain-Specific: 도메인 규칙(법률 조·항, 의료 섹션)으로 분할
- Summarization: 원문을 LLM으로 요약한 결과를 청크로 사용
- Extract Candidate Q: 청크마다 예상 질문을 LLM으로 생성해 함께 저장
- 구조 예시만 확인

In [ ]:
chunk = "신김치를 볶다가 돼지고기를 넣는다. 물을 붓고 20분 끓인다."
llm_result = {   # LLM 호출 결과라고 가정
    "summary": "김치찌개 조리 순서",
    "candidate_questions": ["김치찌개는 몇 분 끓이나요?", "김치찌개 재료는?"],
}
record = {"original": chunk, **llm_result}
print(record)

---

## 주요 청킹(Chunking) 전략의 분할 방식

- 실습 내용
  - 105자를 20자로 나누면 마지막은 5자
  - 제목을 중간 청크에 붙이기
  - 겹침으로 경계 보존
  - 원문 외부 저장 + 포인터

### ① Fixed Size
- 길이·내용 무관 일정 크기 유지
- 105자 → 20자 단위 → 마지막 청크 5자

In [ ]:
text = "가" * 105
chunks = [text[i:i+20] for i in range(0, len(text), 20)]
print([len(c) for c in chunks])

### ② Content Aware
- 구두점·줄바꿈·마크다운 구조 기반 분할
- 문서 제목을 중간 청크에도 추가해 손실 방지

In [ ]:
doc = """# 김치찌개 레시피
신김치를 볶는다.
돼지고기를 넣는다.
물을 붓고 20분 끓인다."""

title, *lines = doc.split("\n")
chunks = [f"{title} | {line}" for line in lines]
for c in chunks: print(c)

### ③ Overlapping
- 겹치는 부분을 두어 쿼리가 한 청크에 온전히 없어도 검색 가능

In [ ]:
text = "신김치를 볶다가 돼지고기를 넣고 물을 부어 끓인 뒤 두부와 파를 마지막에 넣는다"
size, overlap = 15, 5
chunks = [text[i:i+size] for i in range(0, len(text), size - overlap)]
for c in chunks: print(repr(c))

### ④ Recursive
- 원하는 크기가 안 나오면 다른 구분 기호로 재분할
- Fixed + Content Aware 혼합

In [ ]:
def recursive(text, size=12, seps=["\n", ". ", " "]):
    if len(text) <= size or not seps:
        return [text]
    out = []
    for p in text.split(seps[0]):
        out += recursive(p, size, seps[1:]) if len(p) > size else [p]
    return [c for c in out if c.strip()]

doc = "신김치를 볶는다. 돼지고기를 넣고 물을 부어 끓인다.\n두부와 파를 넣는다."
for c in recursive(doc): print(repr(c))

### ⑪ Parent Child
- 청크에는 원문을 저장하지 않고 포인터만 저장
- 원문은 별도 외부 저장소에 보관

In [ ]:
parent_store = {"book1": "김치찌개 챕터 전체 원문 ... (매우 김)"}
child_chunks = [
    {"id": "c1", "text": "신김치를 볶는다", "parent_id": "book1"},
    {"id": "c2", "text": "20분 끓인다", "parent_id": "book1"},
]

hit = child_chunks[1]                           # 검색된 자식 청크
print("검색 결과:", hit["text"])
print("원문 획득:", parent_store[hit["parent_id"]])

---

## Semantic Chunking 상세 동작 구조

- 실습 내용
  - 문장 단위 분할
  - 인접 문장 코사인 유사도 계산
  - 임계값 0.8 기준 통합/분리

In [ ]:
def cos(a, b):
    a, b = np.array(a), np.array(b)
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

In [ ]:
import numpy as np
model = load_model("paraphrase-multilingual-MiniLM-L12-v2")

In [ ]:
doc = ("김치찌개는 신김치와 돼지고기로 만든다. 김치찌개는 얼큰한 국물이 매력이다. 김치찌개에는 두부를 넣으면 좋다. "
       "비빔밥은 나물과 고추장을 비빈다. 비빔밥은 색이 화려하다. "
       "잡채는 당면을 볶아 만든다. 잡채는 명절에 자주 먹는다.")

### 단계 1: 텍스트 분할
- 문장 단위 기초 청크화

In [ ]:
sentences = [s.strip() for s in doc.split(". ") if s.strip()]
sentences[-1] = sentences[-1].rstrip(".")
for i, s in enumerate(sentences): print(i, s)

### 단계 2: 벡터 임베딩 및 유사도 연산
- 인접 문장 쌍의 코사인 유사도

In [ ]:
vecs = model.encode(sentences)
sims = [cos(vecs[i], vecs[i+1]) for i in range(len(vecs) - 1)]
for i, s in enumerate(sims):
    print(f"문장{i} ↔ 문장{i+1}: {s:.3f}")

### 단계 3: 조건부 통합 및 분리
- 유사도 ≥ 임계값 → 같은 청크로 통합
- 유사도 < 임계값 → 새 청크 시작

In [ ]:
def semantic_chunk(sentences, sims, threshold):
    chunks, cur = [], [sentences[0]]
    for i, s in enumerate(sims):
        if s >= threshold:
            cur.append(sentences[i+1])
        else:
            chunks.append(" ".join(cur)); cur = [sentences[i+1]]
    chunks.append(" ".join(cur))
    return chunks

for c in semantic_chunk(sentences, sims, threshold=0.5):
    print("-", c)

### 4. 임계값에 따른 청크 수 변화

In [ ]:
for th in [0.3, 0.5, 0.7, 0.8, 0.9]:
    print(f"임계값 {th}: 청크 {len(semantic_chunk(sentences, sims, th))}개")

---

## 주요 임베딩 모델별 입력 길이 제한

- 실습 내용
  - BERT 토크나이저로 'understanding' 분할
  - 모델별 max_seq_length 비교
  - 토큰 기준 청크 크기 정의

In [ ]:
from transformers import AutoTokenizer

### 1. 토큰 ≠ 단어
- 하나의 단어가 여러 토큰으로 분할

In [ ]:
bert_tok = AutoTokenizer.from_pretrained("bert-base-uncased")
print(bert_tok.tokenize("understanding"))
print(bert_tok.tokenize("kimchi stew is delicious"))

ko_tok = AutoTokenizer.from_pretrained("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
print(ko_tok.tokenize("김치찌개는 맛있다"))

### 2. 모델별 최대 입력 토큰
- BERT/RoBERTa 계열: 512 토큰
- 문장 임베딩 모델은 더 작게 설정된 경우도 있음

In [ ]:
print("bert-base 최대 토큰   :", bert_tok.model_max_length)
for name in ["paraphrase-multilingual-MiniLM-L12-v2", "all-MiniLM-L6-v2"]:
    m = load_model(name)
    print(f"{name:40s} max_seq_length = {m.max_seq_length}")

### 3. 한도 초과 시 잘림 확인

In [ ]:
model = load_model("paraphrase-multilingual-MiniLM-L12-v2")
text = "김치찌개는 신김치와 돼지고기를 넣고 끓인다. " * 40
n_tokens = len(model.tokenizer.tokenize(text))
print("입력 토큰 수:", n_tokens, "/ 한도:", model.max_seq_length)
print("초과 토큰(손실):", max(0, n_tokens - model.max_seq_length))

### 4. 토큰 기준 청크 크기 정의
- '몇 글자'가 아니라 '몇 토큰'

In [ ]:
def chunk_by_tokens(text, tokenizer, max_tokens):
    ids = tokenizer.encode(text, add_special_tokens=False)
    return [tokenizer.decode(ids[i:i+max_tokens]) for i in range(0, len(ids), max_tokens)]

chunks = chunk_by_tokens(text, model.tokenizer, model.max_seq_length - 2)
print("청크 수:", len(chunks))
print("첫 청크 토큰 수:", len(model.tokenizer.tokenize(chunks[0])))